# Agentic Portfolio Construction — Full Pipeline Demo
**Fordham MSQF Capstone 2026**

Five AI agents working in sequence to build a personalized portfolio recommendation:

```
Research Agent  →  Profile Agent  →  Allocation Agent
                                           ↕ (FLAG loop, max 3x)
                                       Risk Agent  ←  Regime Detection
                                           ↓
                                    Compliance Agent
                                           ↓
                                     AdvisorPackage
```

**Every number is deterministic** — pandas, numpy, FRED data, closed-form formulas.  
The LLM writes rationale text and reasoning traces only — it never invents a number.

---
### What's new in this version
- **Regime-adaptive risk caps** — 60-day rolling vol ratio classifies market as NORMAL / ELEVATED / CRISIS.  
  Drawdown caps widen 25 % (ELEVATED) or 50 % (CRISIS) to prevent procyclical selling at market bottoms.
- **Risk-profile downgrade loop** — CRITICAL stress breach steps AGGRESSIVE → MODERATE → CONSERVATIVE  
  one notch per FLAG iteration; reverts automatically on a fresh run.
- **Compliance regime labels fixed** — Compliance Check 1.1a now uses the same academic taxonomy  
  as the Research Agent (`Early Recovery`, `Moderate Expansion`, etc.).

---
### Prerequisites
1. `pip install -r requirements.txt`
2. **First run only** — populate the data cache: run Section 1 below (~10 min, requires WRDS login)
3. Optional env vars:
   - `FRED_API_KEY` — free at fred.stlouisfed.org; falls back to 4.4 % DGS10 if missing
   - `ANTHROPIC_API_KEY` — Claude API; LLM rationale cells are skipped gracefully if missing

---
## 0. Environment Setup

In [ ]:
import os, sys, importlib, logging
from pathlib import Path

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load .env file if present — pip install python-dotenv
try:
    from dotenv import load_dotenv
    load_dotenv(Path(PROJECT_ROOT) / '.env')
    print('.env loaded')
except ImportError:
    print('dotenv not installed — run: pip install python-dotenv')

importlib.invalidate_caches()
logging.basicConfig(level=logging.WARNING)

FRED_KEY      = os.environ.get('FRED_API_KEY')
ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY')
print(f'FRED_API_KEY set:      {bool(FRED_KEY)}')
print(f'ANTHROPIC_API_KEY set: {bool(ANTHROPIC_KEY)}')

---
## 1. One-Time Data Cache Setup

The pipeline reads from `data/storage/` (gitignored, ~500 MB).  
Run once to populate, then skip this section.

| File | Source | Used by |
|---|---|---|
| `crsp_monthly.parquet` | WRDS/CRSP | Allocation — BL market-cap prior weights |
| `crsp_daily.parquet` | WRDS/CRSP | Risk — VaR, drawdown, regime detection |
| `ff_risk_factors.parquet` | WRDS/FF | Allocation + Risk — 4-factor model |
| `ff12_monthly.parquet` | Ken French | Research — regime feature validation |
| `bls_oes.parquet` | BLS OES 2023 | Profile — salary distributions |
| `fred_macro.parquet` | FRED | Research — 13 macro series |
| `permno_map.json` | WRDS | Risk — ticker → CRSP permno |
| `mkt_cap_weights.json` | WRDS | Allocation — BL equilibrium prior |

In [ ]:
from pathlib import Path

REQUIRED = [
    'crsp_monthly.parquet', 'crsp_daily.parquet', 'ff_risk_factors.parquet',
    'fred_macro.parquet',   'bls_oes.parquet',    'ff12_monthly.parquet',
    'permno_map.json',      'mkt_cap_weights.json',
]
storage = Path('data/storage')

missing = [f for f in REQUIRED if not (storage / f).exists()]
if not missing:
    print('All data files present — skip to Section 2.')
else:
    for f in REQUIRED:
        mark = 'OK     ' if (storage / f).exists() else 'MISSING'
        print(f'  {mark}  {f}')

In [ ]:
# WRDS connection — prompts for username + password on first run
from data.fetch.wrds import get_connection
conn = get_connection()
print('Connected to WRDS')

In [ ]:
from agents.allocation.adapters import DEFAULT_TICKERS
from data.fetch.wrds import fetch_crsp_monthly, fetch_crsp_daily, fetch_ff_factors

fetch_crsp_monthly(DEFAULT_TICKERS, conn=conn)   # crsp_monthly.parquet + permno_map + mkt_cap_weights
fetch_crsp_daily(DEFAULT_TICKERS, conn=conn)     # crsp_daily.parquet  (~5 min)
fetch_ff_factors(conn=conn)                      # ff_risk_factors.parquet
print('WRDS fetch done')

In [ ]:
from data.fetch.fred import fetch_fred_macro
from data.fetch.bls  import fetch_bls_oes
from data.fetch.factors import fetch_ff12

fetch_fred_macro(fred_api_key=FRED_KEY)
fetch_bls_oes()
fetch_ff12()
print('Public data fetch done')

---
## 2. Agent 1 — Research Agent: Macro Regime Detection

**What it does:**  
Pulls 13 FRED macro series (yield curve, unemployment, CPI, credit spread, …), runs PELT  
change-point detection to find structural breaks, clusters segments with K-means, then  
trains an XGBoost classifier against five historically-labelled anchor windows:

| Label | Anchor window | Macro signature |
|---|---|---|
| Early Recovery | 2003-06 – 2004-06 | Low rates, spreads narrowing post dot-com |
| Late-Cycle Expansion | 2005-01 – 2007-06 | Rising rates, low VIX, pre-GFC boom |
| Financial Crisis & ZLB | 2008-09 – 2010-12 | Acute GFC + zero interest rate policy |
| Moderate Expansion | 2015-01 – 2019-06 | Post-QE normalisation, stable growth |
| Inflation Shock | 2022-01 – 2023-06 | Peak CPI, aggressive Fed tightening |

A 6-month rolling majority vote smooths month-to-month noise at regime boundaries.

**Output:** `MacroRegimeSnapshot` — passed unchanged to every downstream agent.

In [ ]:
from agents.research.research_agent import run_research_agent

# compare_models=False skips HMM/GMM benchmarking (paper validation only)
macro = run_research_agent(fred_api_key=FRED_KEY, compare_models=False)

print(f'Regime:          {macro.regime_label}')
print(f'Confidence:      {macro.regime_confidence:.0%}')
print(f'Prior regime:    {macro.prior_regime}')
print(f'Regime change:   {macro.regime_change_detected}')
print(f'Low confidence:  {macro.is_low_confidence}')
print(f'As of:           {macro.as_of}')
print()
print('--- Key FRED signals ---')
print(f'Yield curve (10Y-2Y):  {macro.yield_curve:+.2f} pp')
print(f'Fed funds rate:         {macro.fed_funds:.2f}%')
print(f'Unemployment:           {macro.unemployment:.1f}%')
print(f'CPI YoY:                {macro.cpi:.1f}%')
print(f'Credit spread:          {macro.credit_spread:.2f} pp')

---
## 3. Agent 2 — Profile Agent: Human Capital Valuation

**What it does:**  
Maps BLS OES May 2023 salary data to 9 occupational personas, looks up income-equity  
beta (β) and correlation (ρ) from a calibrated table (Ibbotson et al. 2007), then computes:

| Formula | Meaning |
|---|---|
| `HC = Salary × [1 − (1+r)^{−n}] / r` | PV of future earnings (annuity, DGS10 discount) |
| `implicit_equity_exposure = hc_share × β` | Equity risk already carried through the career |
| `effective_risk_budget = (FC + HC×(1−σ)) / total_wealth` | Total risk capacity across the balance sheet |
| `portfolio_equity_target = risk_budget − implicit_equity_exposure` | Equity headroom for the investment portfolio |

A tech exec with β = 1.2 already has ~90 % implicit equity exposure through career + RSUs —  
their portfolio should be mostly bonds. A tenured professor with β = 0.05 can hold a much  
higher equity allocation.

**Output:** `list[ProfileAgentOutput]` — 9 validated Pydantic objects, one per BLS occupation.

In [ ]:
from agents.profile.profile_agent import run_profile_agent
import pandas as pd

profiles = run_profile_agent(fred_api_key=FRED_KEY)
print(f'Built {len(profiles)} profiles')
print()

summary = pd.DataFrame([{
    'Client':          p.client_id,
    'HC type':         p.human_capital_type.value,
    'β':               f'{p.income_equity_beta:.2f}',
    'Implicit eq exp': f'{p.implicit_equity_exposure:.1%}',
    'Risk budget':     f'{p.effective_risk_budget:.1%}',
    'HC % wealth':     f'{p.human_capital_pct_of_total:.0f}%',
} for p in profiles])
print(summary.to_string(index=False))

In [ ]:
# Pick one persona to run through the rest of the pipeline
# SOC 15-1252 = Software Developers (equity-like HC, β ≈ 1.2)
profile = next(p for p in profiles if p.client_id == 'bls_15-1252_p50')

print(f'Selected:               {profile.client_id}')
print(f'Career type:            {profile.career_type}')
print(f'Age / horizon:          {profile.age} yrs / {profile.investment_horizon_years} yrs')
print(f'Financial capital:      ${profile.financial_capital:>12,.0f}')
print(f'Human capital (PV):     ${profile.human_capital_valuation:>12,.0f}')
print(f'Total wealth:           ${profile.total_wealth:>12,.0f}')
print(f'HC type:                {profile.human_capital_type.value}')
print(f'Income beta (β):        {profile.income_equity_beta:.2f}')
print(f'Income-equity corr (ρ): {profile.income_equity_correlation:.2f}')
print(f'Income volatility (σ):  {profile.income_volatility_sigma:.2f}')
print(f'Implicit equity exp:    {profile.implicit_equity_exposure:.1%}')
print(f'Effective risk budget:  {profile.effective_risk_budget:.1%}')
print(f'Risk tolerance:         {profile.risk_tolerance_level.value}')
print(f'Employer sector:        {profile.industry_exposure_sector}')

---
## 4. Agents 3 + 4 — Allocation ↔ Risk Loop

**Allocation Agent:**  
Uses Black-Litterman starting from CRSP market-cap equilibrium weights, runs a 4-factor  
(MKT/SMB/HML/UMD) model to estimate expected returns, then subtracts `implicit_equity_exposure`  
so total (career + portfolio) equity risk stays appropriate for the client.

**Risk Agent:**  
Runs deterministic stress tests and returns `APPROVE`, `FLAG`, or `REJECT`.

**New — Regime-Adaptive Risk Caps:**  
Before computing drawdown thresholds, the Risk Agent classifies the current market regime  
from 60-day rolling portfolio return volatility vs the full-sample baseline:

| Regime | Vol ratio trigger | Drawdown cap multiplier | Example period |
|---|---|---|---|
| NORMAL | < 1.5× | 1.00× (standard) | 2012–2019 |
| ELEVATED | 1.5× – 2.0× | 1.25× (25% wider) | H2 2007, 2011 |
| CRISIS | ≥ 2.0× | 1.50× (50% wider) | Oct 2008, Mar 2020 |

Widening the cap during a crisis prevents the system from forcing a sell at market  
bottoms — the cap resets automatically once volatility normalises.

**FLAG loop (max 3 iterations):**  
On FLAG, violated constraints are fed back to the Allocation Agent which re-optimises  
with tighter limits. A CRITICAL stress breach steps the risk profile down one notch  
(AGGRESSIVE → MODERATE → CONSERVATIVE); the downgrade reverts on a fresh run.

In [ ]:
import logging
logging.getLogger().setLevel(logging.INFO)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

from agents.orchestrator.orchestrator import run_pipeline

pkg = run_pipeline(profile, macro, fred_api_key=FRED_KEY)

logging.getLogger().setLevel(logging.WARNING)
print('\nPipeline complete.')

---
## 5. Results

In [ ]:
# Pipeline metadata
m = pkg.metadata
print('=== PIPELINE METADATA ===')
print(f'  Final risk decision:      {m.final_risk_decision.value}')
print(f'  Final compliance status:  {m.final_compliance_status.value}')
print(f'  Risk FLAG revisions:      {m.risk_revisions}')
print(f'  Compliance revisions:     {m.compliance_revisions}')
if m.pipeline_warnings:
    print('  Warnings:')
    for w in m.pipeline_warnings:
        print(f'    - {w}')

In [ ]:
# Portfolio weights
weights = pkg.allocation.proposed_portfolio
df_w = pd.DataFrame(
    [{'Ticker': t, 'Weight': w, 'Weight %': f'{w:.1%}'}
     for t, w in sorted(weights.items(), key=lambda x: x[1], reverse=True)
     if w > 0.001]
)
print('=== PORTFOLIO WEIGHTS ===')
print(df_w.to_string(index=False))
print(f'\nTotal: {sum(weights.values()):.4f}')

In [ ]:
# Risk Agent output — includes new regime-adaptive fields
r = pkg.risk
print('=== RISK AGENT OUTPUT ===')
print(f'Decision:    {r.risk_decision.value}')

# ── Regime-adaptive cap (new) ──────────────────────────────────────────
if r.market_regime is not None:
    MULTIPLIERS = {'normal': 1.00, 'elevated': 1.25, 'crisis': 1.50}
    mult = MULTIPLIERS.get(r.market_regime.value, 1.0)
    cap_note = {
        'normal':   'standard cap',
        'elevated': 'widened 25% — elevated vol',
        'crisis':   'widened 50% — crisis vol',
    }.get(r.market_regime.value, '')
    print(f'Market regime:         {r.market_regime.value.upper()}  ({cap_note})')
    print(f'Effective drawdown cap: {r.effective_drawdown_cap:.2%}')

if r.portfolio_volatility_annual is not None:
    print(f'Volatility (annual):    {r.portfolio_volatility_annual:.2%}')

print(f'Violations: {r.violations if r.violations else "None"}')

# ── Regime stress tests (Research Agent taxonomy) ──────────────────────
print('\nRegime stress tests (portfolio loss vs benchmark):')
for regime, ev in r.regime_evaluation.items():
    status = 'PASS' if ev.passed else 'FAIL'
    print(f'  [{status}] {regime:<32}  portfolio {ev.portfolio_drawdown:.1%}  bench {ev.benchmark_drawdown:.1%}')

In [ ]:
# Compliance Agent output
c = pkg.compliance
print('=== COMPLIANCE AGENT OUTPUT ===')
print(f'Clearance:        {c.clearance}')
print(f'Status:           {c.compliance_status.value}')
print(f'Overall severity: {c.overall_severity.value}')
print(f'Recommendation:   {c.recommendation}')

if c.passed_checks:
    sample = ', '.join(c.passed_checks[:5])
    suffix = '...' if len(c.passed_checks) > 5 else ''
    print(f'\nPassed checks ({len(c.passed_checks)}): {sample}{suffix}')

if c.violations:
    print(f'\nViolations ({len(c.violations)}):')
    for v in c.violations:
        print(f'  [{v.severity.value}] {v.check}: {v.description}')

In [ ]:
# LLM-generated allocation rationale (sample ticker)
rationale = pkg.allocation.allocation_rationale
if rationale:
    sample_ticker = next(iter(rationale))
    print(f'=== ALLOCATION RATIONALE — {sample_ticker} ===')
    print(rationale[sample_ticker])
else:
    print('No rationale (ANTHROPIC_API_KEY not set)')

---
## 6. Regime-Adaptive Cap — Walkthrough

This section shows how the cap changes across regimes for the selected client,  
independent of a full pipeline run. Useful for validating the logic and for the thesis writeup.

In [ ]:
from contracts import MarketRegime, RiskProfile
from agents.shared.core.risk import MAX_DRAWDOWN_CAP, effective_cap

base_profile = profile.risk_tolerance_level  # e.g. RiskProfile.AGGRESSIVE

print(f'Client risk profile: {base_profile.value}')
print(f'Base drawdown cap:   {MAX_DRAWDOWN_CAP[base_profile]:.0%}')
print()
print(f'{"Regime":<12}  {"Multiplier":<12}  {"Effective cap":<14}  Example period')
print('-' * 65)

examples = {
    MarketRegime.NORMAL:   '2012 – 2019 bull market',
    MarketRegime.ELEVATED: 'H2 2007, Aug 2011 downgrade',
    MarketRegime.CRISIS:   'Oct 2008, Mar 2020 COVID',
}
multipliers = {MarketRegime.NORMAL: 1.00, MarketRegime.ELEVATED: 1.25, MarketRegime.CRISIS: 1.50}

for regime in MarketRegime:
    cap  = effective_cap(base_profile, regime)
    mult = multipliers[regime]
    print(f'{regime.value:<12}  {mult:<12.2f}  {cap:<14.2%}  {examples[regime]}')

In [ ]:
# Show caps across all three risk profiles × all three regimes
print(f'{"":<12}', end='')
for regime in MarketRegime:
    print(f'  {regime.value.upper():<14}', end='')
print()
print('-' * 58)

for rp in RiskProfile:
    print(f'{rp.value:<12}', end='')
    for regime in MarketRegime:
        cap = effective_cap(rp, regime)
        print(f'  {cap:<14.2%}', end='')
    print()

---
## 7. Batch Run — All 9 BLS Personas

`run_all()` produces one `AdvisorPackage` per BLS occupation persona.  
The Research Agent runs once; the Profile → Allocation → Risk → Compliance chain runs 9 times.

In [ ]:
from agents.orchestrator.orchestrator import run_all

# Pass the already-computed macro snapshot to skip the Research Agent re-run
packages = run_all(fred_api_key=FRED_KEY, macro=macro)

print(f'{len(packages)} AdvisorPackages produced')
print()

rows = []
for p in packages:
    r = p.risk
    regime_str = r.market_regime.value.upper() if r.market_regime else 'N/A'
    rows.append({
        'Client':       p.profile.client_id,
        'HC type':      p.profile.human_capital_type.value,
        'Risk':         p.metadata.final_risk_decision.value,
        'Compliance':   p.metadata.final_compliance_status.value,
        'Regime':       regime_str,
        'DD cap':       f'{r.effective_drawdown_cap:.0%}' if r.effective_drawdown_cap else 'N/A',
        'FLAG revs':    p.metadata.risk_revisions,
        'Top holding':  max(p.allocation.proposed_portfolio,
                            key=p.allocation.proposed_portfolio.get),
    })

print(pd.DataFrame(rows).to_string(index=False))